# Kinan Math Animation Studio: ganti suara narator dengan suara Anda (Seed-VC)

Notebook ini mengubah **hanya suara narator** di empat video yang sudah jadi. Gambar, caption, animasi, urutan adegan, dan durasi tidak disentuh (video disalin tanpa encode ulang, lalu dibuktikan identik).

**Privasi:** rekaman suara Anda hanya diunggah ke sesi Colab ini. Tidak dikirim ke GitHub. Setelah selesai, jalankan sel terakhir untuk menghapusnya.

**Cara pakai:** *Runtime → Change runtime type → T4 GPU*, lalu jalankan sel dari atas ke bawah.

> Status: notebook ini belum pernah dijalankan oleh pembuatnya (sesi pembuatnya tidak punya GPU dan tidak bisa mengakses Hugging Face). Bagian kode yang bergantung model Seed-VC belum teruji.

In [ ]:
import torch, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (akan lambat: ganti Runtime ke T4 GPU)")

In [ ]:
%%bash
set -e
git clone --depth 1 --branch claude/math-animation-studio-577wfc https://github.com/ufukkinanti22-sys/kinan-math-animation-studio.git kinan
git clone --depth 1 https://github.com/Plachtaa/seed-vc.git kinan/seed-vc
apt-get -qq install -y ffmpeg > /dev/null
pip -q install -r kinan/voice_swap/requirements.txt

In [ ]:
%cd /content/kinan
from google.colab import files
print("Unggah Kinan_Voice_References.zip (rekaman suara Anda):")
up = files.upload()
ZIP = "/content/kinan/" + next(iter(up))

## 1. Percobaan pada video P01
Memilih klip suara paling bersih, mengonversi 21 segmen narasi, memverifikasi (kelengkapan kata, waktu, kemiripan suara), lalu membuat MP4 baru.

In [ ]:
!python -m voice_swap.run --zip "$ZIP" --seedvc-dir seed-vc --stage trial

In [ ]:
import json, IPython.display as ipd
rep = json.load(open("voice_out/reports/p01_dadu_berbeda.json"))
print("Lolos verifikasi:", rep["verified"], "| gambar identik dengan video asli:", rep["picture_identical_to_original"])
print(json.dumps(rep["verification"]["summary"], indent=1))
print("Masalah:", rep["verification"]["problems"])
# dengarkan beberapa segmen hasil konversi
for beat in ["hook", "options", "fraction"]:
    print(beat); ipd.display(ipd.Audio(f"voice_private/converted/p01_dadu_berbeda/{beat}.wav"))

## 2. Tiga video lainnya (hanya jika percobaan P01 lolos verifikasi)

In [ ]:
!python -m voice_swap.run --zip "$ZIP" --seedvc-dir seed-vc --stage rest

In [ ]:
from google.colab import files
files.download("voice_out/Kinan_Voice_Swap_Hasil.zip")   # 4 MP4 + WAV per video + WAV per segmen + laporan

## 3. Bersihkan (disarankan)
Menghapus klip referensi dan semua salinan suara dari sesi ini. Setelah itu pilih *Runtime → Disconnect and delete runtime*.

In [ ]:
!rm -rf /content/kinan/voice_private /content/kinan/*.zip